# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI
import os

document_folder = "../../05_src/documents/"
blue_cross_file = "the_blue_cross.txt"
file_path = os.path.join(document_folder, blue_cross_file)

with open(file_path, "r", encoding="utf-8") as f:
    blue_cross_text = f.read()

In [3]:
instructions = "You are an helpful assistant that summarizes works of fiction with a quirky and bubbly approach."
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [ ]:
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.2
)

In [5]:
response.output_text

"Hello, dear reader! 🎉 Are you ready for an enchanting adventure filled with mystery, clever criminals, and the surprising world of detective work? Let’s dive into “The Blue Cross” by G.K. Chesterton, where a notable pursuit unfolds that is as intricate as a set of nested dolls! 🌟\n\nIn this tantalizing story, we meet the unassuming yet brilliant detective, Aristide Valentin, who arrives in London on a mission to catch the notorious criminal, Flambeau. Clad in somewhat flamboyant clothes, he blends into the bustling crowd but carries a hidden revolver and a police card—oh my! Flambeau, a giant of mischief known for elaborate robberies, cleverly maneuvers through London, likely disguising himself among the festivities of the Eucharistic Congress. Valentin keenly observes, acknowledging that Flambeau's height is his only unsolvable puzzle.\n\nWhile Valentin’s investigative journey leads him through quirky encounters, like the flustered priest with brown paper parcels and an intriguing li

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [9]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    _openai_api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text,
    
)

In [10]:
metric.measure(test_case)

Output()

0.6896551724137931

In [11]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 0.6896551724137931

**Reason**: The score is 0.69 because while the summary captures some key elements of the story, it includes several irrelevant details that do not contribute to the overall understanding of the plot or characters. These irrelevant statements detract from the clarity and focus of the summary, preventing a higher score.

# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [12]:
instructions = "You are an helpful assistant that specializes in works of fiction."
PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [13]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=0.7
)

In [14]:
response.output_text

"The main antagonist in the story is Flambeau, a notorious criminal known for his ingenious and audacious thefts. His motivation for his actions primarily revolves around the thrill of committing clever crimes and the pursuit of valuable objects, such as the silver cross with sapphires that Father Brown possesses. Flambeau's personality is characterized by a blend of physical prowess and cunning intelligence, and he enjoys outsmarting authorities, which drives him to continuously challenge the law and engage in elaborate schemes. Ultimately, his actions are motivated by a combination of greed, the excitement of crime, and the desire to showcase his cleverness by evading capture."

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [15]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 1 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 2 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 3 time(s)...

...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 10 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 11 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 12 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 13 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'message': 'Limit Exceeded'} Retrying: 14 time(s)...

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [ ]:
...

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])

In [ ]:
result.model_dump()